# ScrapLink — Sprint 2: Price Recommendation Model Training
**AI-Driven B2B Industrial Scrap Exchange Platform**  
**Team:** Daksh Dashora (ML Lead) | Anushka Kumawat | Arman Agrawal  
**Sprint 2 Goal:** Train a baseline Linear Regression + XGBoost price recommender on engineered MCX commodity features.

---
### Notebook Structure
1. Install Dependencies
2. Configuration & Data Loading
3. Exploratory Data Analysis (EDA)
4. Feature Engineering Review
5. Model Training — Linear Regression Baseline
6. Model Training — XGBoost Regressor
7. Evaluation & Comparison
8. Feature Importance
9. Save Model Artifacts
10. Live Inference Demo

## 1. Install Dependencies

In [ ]:
# Run this cell once on Kaggle / Colab
import subprocess, sys

packages = ['xgboost', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'joblib']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print('All packages ready!')

## 2. Configuration & Data Loading

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5)})

print('Imports OK')

In [ ]:
# ─── CONFIG ───────────────────────────────────────────────────────────────────
# Update FEATURE_CSV to the path where your engineered_commodity_features.csv lives
# On Kaggle  : upload the CSV to the dataset input and set the path below
# On Colab   : upload via Files panel or mount Drive and set the path below

FEATURE_CSV = 'engineered_commodity_features.csv'   # <── CHANGE THIS if needed
MODELS_DIR  = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

FEATURE_COLS = [
    'commodity_encoded',
    'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'rolling_mean_7', 'rolling_mean_30',
    'rolling_std_7',  'rolling_std_30',
    'pct_change_7',   'pct_change_30',
]
TARGET_COL = 'spot_price_inr_kg'
print('Config OK')

In [ ]:
# ─── Load Dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv(FEATURE_CSV)
df['date'] = pd.to_datetime(df['date'])

print(f'Shape          : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range     : {df.date.min().date()} to {df.date.max().date()}')
print(f'Commodities    : {sorted(df.commodity.unique())}')
print(f'Missing values :\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head(3)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics per commodity
stats = (
    df.groupby('commodity')[TARGET_COL]
    .agg(['mean','std','min','median','max'])
    .round(2)
    .rename(columns={'mean': 'Mean (Rs/kg)', 'std': 'Std Dev', 'min': 'Min',
                     'median': 'Median', 'max': 'Max'})
)
stats['Volatility%'] = (stats['Std Dev'] / stats['Mean (Rs/kg)'] * 100).round(1)
print('=== Commodity Price Statistics ===')
stats

In [ ]:
# Price time series per commodity
commodities = sorted(df.commodity.unique())
n = len(commodities)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, comm in enumerate(commodities):
    sub = df[df.commodity == comm].sort_values('date')
    axes[i].plot(sub['date'], sub[TARGET_COL], linewidth=1.2, color='steelblue')
    axes[i].fill_between(sub['date'], sub[TARGET_COL], alpha=0.15, color='steelblue')
    axes[i].set_title(comm, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Rs/kg')
    axes[i].xaxis.set_major_locator(plt.MaxNLocator(4))
    axes[i].tick_params(axis='x', rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('ScrapLink — Commodity Spot Price Time Series (INR/kg)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'price_timeseries.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Volatility bar chart
vol = stats['Volatility%'].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(vol.index, vol.values, color=sns.color_palette('muted', len(vol)))
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_xlabel('Volatility (Std/Mean × 100)')
ax.set_title('Price Volatility by Commodity — Justifies Rolling Std Features for XGBoost', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'volatility_chart.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Pivot price per commodity and show correlation matrix
pivot = df.pivot_table(index='date', columns='commodity', values=TARGET_COL, aggfunc='mean')
corr = pivot.corr().round(2)

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Commodity Price Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'correlation_heatmap.png'), bbox_inches='tight')
plt.show()

## 4. Feature Engineering Review

In [ ]:
# Show the engineered features for one commodity
sample = df[df.commodity == 'Copper'].sort_values('date').tail(20)
display_cols = ['date', 'spot_price_inr_kg', 'lag_1', 'lag_7', 'lag_30',
                'rolling_mean_7', 'rolling_std_7', 'pct_change_7']
print('=== Engineered Features Sample (Copper - last 20 rows) ===')
sample[display_cols].reset_index(drop=True)

In [ ]:
# Feature correlation with target
le_temp = LabelEncoder()
df['commodity_encoded'] = le_temp.fit_transform(df['commodity'])
df_clean = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)

feat_corr = df_clean[FEATURE_COLS + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL).sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['tomato' if v < 0 else 'steelblue' for v in feat_corr]
feat_corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with spot_price_inr_kg')
ax.set_title('Feature Correlation with Target', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'feature_correlation.png'), bbox_inches='tight')
plt.show()

## 5. Train-Test Split (Time-Aware)

In [ ]:
# Sort by date to avoid data leakage — NO random shuffle
df_clean = df_clean.sort_values('date').reset_index(drop=True)

X = df_clean[FEATURE_COLS].values
y = df_clean[TARGET_COL].values

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Scale for Linear Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training rows : {len(X_train):,}')
print(f'Test rows     : {len(X_test):,}')
print(f'Split date    : {df_clean.iloc[split_idx]["date"].date()}')

## 6. Model 1 — Linear Regression Baseline

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = float(np.sqrt(mean_squared_error(y_test, y_pred_lr)))
ss_res  = np.sum((y_test - y_pred_lr)**2)
r2_lr   = 1 - ss_res / np.sum((y_test - y_test.mean())**2)

print('=== Linear Regression (Baseline) ===')
print(f'MAE  : Rs {mae_lr:.2f}/kg')
print(f'RMSE : Rs {rmse_lr:.2f}/kg')
print(f'R2   : {r2_lr:.4f}')

## 7. Model 2 — XGBoost Regressor

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

# XGBoost does NOT need scaled features
xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_pred_xgb = xgb.predict(X_test)
mae_xgb    = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb   = float(np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
r2_xgb     = 1 - np.sum((y_test - y_pred_xgb)**2) / np.sum((y_test - y_test.mean())**2)

print('=== XGBoost Regressor (Sprint 2 Model) ===')
print(f'MAE  : Rs {mae_xgb:.2f}/kg')
print(f'RMSE : Rs {rmse_xgb:.2f}/kg')
print(f'R2   : {r2_xgb:.4f}')

## 8. Evaluation & Visual Comparison

In [ ]:
results = pd.DataFrame([
    {'Model': 'Linear Regression (Baseline)', 'MAE (Rs/kg)': mae_lr,  'RMSE (Rs/kg)': rmse_lr,  'R2': r2_lr},
    {'Model': 'XGBoost Regressor (Sprint 2)',  'MAE (Rs/kg)': mae_xgb, 'RMSE (Rs/kg)': rmse_xgb, 'R2': r2_xgb},
]).round(2).set_index('Model')

print('=== Sprint 2 — Model Evaluation Summary ===')
results

In [ ]:
# MAE & RMSE comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
models = ['Linear Regression', 'XGBoost']
maes   = [mae_lr,  mae_xgb]
rmses  = [rmse_lr, rmse_xgb]
colors = ['#6baed6', '#2171b5']

for ax, vals, metric in zip(axes, [maes, rmses], ['MAE (Rs/kg)', 'RMSE (Rs/kg)']):
    bars = ax.bar(models, vals, color=colors, edgecolor='white', width=0.5)
    ax.bar_label(bars, fmt='Rs %.2f', padding=4, fontsize=10)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.3)
    ax.set_ylabel(metric)

fig.suptitle('Linear Regression vs XGBoost — Metric Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'model_comparison_bar.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Predicted vs Actual scatter plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, name, color in zip(
    axes,
    [y_pred_lr, y_pred_xgb],
    ['Linear Regression', 'XGBoost'],
    ['#6baed6', '#2171b5'],
):
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.scatter(y_test, y_pred, alpha=0.25, s=10, color=color)
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual Price (Rs/kg)')
    ax.set_ylabel('Predicted Price (Rs/kg)')
    ax.set_title(f'{name} — Actual vs Predicted', fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'actual_vs_predicted.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, name, color in zip(
    axes,
    [y_pred_lr, y_pred_xgb],
    ['Linear Regression', 'XGBoost'],
    ['#6baed6', '#2171b5'],
):
    residuals = y_test - y_pred
    ax.scatter(y_pred, residuals, alpha=0.25, s=10, color=color)
    ax.axhline(0, color='red', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Predicted Price (Rs/kg)')
    ax.set_ylabel('Residual (Actual - Predicted)')
    ax.set_title(f'{name} — Residual Plot', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'residual_plots.png'), bbox_inches='tight')
plt.show()

## 9. XGBoost Feature Importance

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_fi = sns.color_palette('Blues_r', len(importances))
importances.plot(kind='barh', ax=ax, color=colors_fi)
ax.set_xlabel('Feature Importance Score')
ax.set_title('XGBoost — Feature Importance (F-score)', fontweight='bold')
for i, v in enumerate(importances):
    ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'feature_importance.png'), bbox_inches='tight')
plt.show()

## 10. Save Model Artifacts

In [ ]:
# Label encoder (refit on clean df)
le = LabelEncoder()
le.fit(df_clean['commodity'])

# Save
joblib.dump(lr,     os.path.join(MODELS_DIR, 'linear_regression_baseline.pkl'))
joblib.dump(xgb,    os.path.join(MODELS_DIR, 'xgboost_price_model.pkl'))
joblib.dump(scaler, os.path.join(MODELS_DIR, 'preprocessor.pkl'))
joblib.dump(le,     os.path.join(MODELS_DIR, 'label_encoder.pkl'))

report = {
    'sprint': 'Sprint 2 - Baseline Price Recommendation',
    'feature_columns': FEATURE_COLS,
    'target': TARGET_COL,
    'train_rows': len(X_train),
    'test_rows':  len(X_test),
    'commodities': sorted(df_clean['commodity'].unique().tolist()),
    'results': [
        {'model': 'Linear Regression', 'MAE': round(mae_lr, 2),  'RMSE': round(rmse_lr, 2),  'R2': round(r2_lr, 4)},
        {'model': 'XGBoost',           'MAE': round(mae_xgb, 2), 'RMSE': round(rmse_xgb, 2), 'R2': round(r2_xgb, 4)},
    ],
    'best_model': 'XGBoost' if rmse_xgb < rmse_lr else 'LinearRegression',
}
with open(os.path.join(MODELS_DIR, 'evaluation_report.json'), 'w') as f:
    json.dump(report, f, indent=2)

print('Artifacts saved:')
for fname in os.listdir(MODELS_DIR):
    path = os.path.join(MODELS_DIR, fname)
    size = os.path.getsize(path)
    print(f'  {fname:<40} {size:>8,} bytes')

## 11. Live Inference Demo — predict_price()

In [ ]:
def predict_price(
    commodity: str,
    lag_1, lag_7, lag_14, lag_30,
    rolling_mean_7, rolling_mean_30,
    rolling_std_7, rolling_std_30,
    pct_change_7, pct_change_30,
    confidence_rmse: float = None,
):
    """Loads saved model and returns price recommendation dict."""
    _xgb = joblib.load(os.path.join(MODELS_DIR, 'xgboost_price_model.pkl'))
    _le  = joblib.load(os.path.join(MODELS_DIR, 'label_encoder.pkl'))

    if confidence_rmse is None:
        confidence_rmse = rmse_xgb   # use evaluation RMSE from this session

    try:
        commodity_enc = _le.transform([commodity])[0]
    except ValueError:
        commodity_enc = int(np.mean(_le.transform(_le.classes_)))

    feats = np.array([[commodity_enc,
                       lag_1, lag_7, lag_14, lag_30,
                       rolling_mean_7, rolling_mean_30,
                       rolling_std_7, rolling_std_30,
                       pct_change_7, pct_change_30]])
    pred  = float(_xgb.predict(feats)[0])
    lower = round(pred - 1.96 * confidence_rmse, 2)
    upper = round(pred + 1.96 * confidence_rmse, 2)
    pred  = round(pred, 2)

    trend = 'UPWARD' if pred > rolling_mean_7 * 1.01 else ('DOWNWARD' if pred < rolling_mean_7 * 0.99 else 'STABLE')

    return {
        'recommended_price': pred,
        'price_band':  {'lower': max(lower, 0.0), 'upper': upper},
        'trend':       trend,
        'unit':        'INR/kg',
        'commodity':   commodity,
        'disclaimer':  'AI-estimated reference price. Not a guaranteed transaction price.',
    }


# --- Demo: predict Copper price ---
copper = df[df.commodity == 'Copper'].sort_values('date').iloc[-1]

result = predict_price(
    commodity       = 'Copper',
    lag_1           = copper['lag_1'],
    lag_7           = copper['lag_7'],
    lag_14          = copper['lag_14'],
    lag_30          = copper['lag_30'],
    rolling_mean_7  = copper['rolling_mean_7'],
    rolling_mean_30 = copper['rolling_mean_30'],
    rolling_std_7   = copper['rolling_std_7'],
    rolling_std_30  = copper['rolling_std_30'],
    pct_change_7    = copper['pct_change_7'],
    pct_change_30   = copper['pct_change_30'],
)

print('=== /api/predict-price Response Demo (Copper) ===')
print(json.dumps(result, indent=2))

In [ ]:
# Predict for all commodities using their latest known features
rows = []
for comm in sorted(df.commodity.unique()):
    latest = df[df.commodity == comm].sort_values('date').dropna(subset=FEATURE_COLS).iloc[-1]
    res = predict_price(
        commodity=comm,
        lag_1=latest['lag_1'], lag_7=latest['lag_7'],
        lag_14=latest['lag_14'], lag_30=latest['lag_30'],
        rolling_mean_7=latest['rolling_mean_7'],  rolling_mean_30=latest['rolling_mean_30'],
        rolling_std_7=latest['rolling_std_7'],    rolling_std_30=latest['rolling_std_30'],
        pct_change_7=latest['pct_change_7'],      pct_change_30=latest['pct_change_30'],
    )
    rows.append({
        'Commodity':           comm,
        'Recommended (Rs/kg)': res['recommended_price'],
        'Lower Band (Rs/kg)':  res['price_band']['lower'],
        'Upper Band (Rs/kg)':  res['price_band']['upper'],
        'Trend':               res['trend'],
    })

pred_df = pd.DataFrame(rows).set_index('Commodity')
print('=== Live Recommendations for All Commodities ===')
pred_df